# 1. Get to know the problem

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
              Path('/kaggle/input/rsna-knee-abnormality-detection'), Path('upload'), Path('.')]
ROOT = next((p for p in candidates if (p/'train.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('File not found.')
names = ['train', 'train_series', 'test', 'test_series', 'sample_submission']
tables = {n: pd.read_csv(ROOT / (n+'.csv')) for n in names}
train, series, test, test_series, submission = [tables[n] for n in names]
ID, SID = 'StudyInstanceUID', 'SeriesInstanceUID'
LABELS = [c for c in submission if c != ID]
print('Nguồn:', ROOT)
display(pd.DataFrame([{'file': n, 'rows': len(d), 'columns': len(d.columns),
                      'duplicate_rows': int(d.duplicated().sum()),
                      'missing_cells': int(d.isna().sum().sum())} for n,d in tables.items()]))

In [ ]:
y = train[LABELS]
known = y.notna().sum(axis=1)
status = pd.Series(np.select([known.eq(0), known.eq(len(LABELS))],
                            ['unlabeled','fully_labeled'], default='partially_labeled'), index=train.index)
display(status.value_counts().rename('studies').to_frame())

In [ ]:
label_summary = pd.DataFrame({'known': y.notna().sum(), 'missing': y.isna().sum(),
                              'negative': y.eq(0).sum(), 'positive': y.eq(1).sum()})
label_summary['invalid_nonbinary'] = ((~y.isin([0,1])) & y.notna()).sum()
label_summary['positive_pct_among_known'] = (100*label_summary.positive/label_summary.known.replace(0,np.nan)).round(2)
display(label_summary)

In [ ]:
fully_labeled = train.loc[status.eq('fully_labeled')].copy()
print('Study all labeled 0:', int(fully_labeled[LABELS].sum(axis=1).eq(0).sum()))
print('Distribution of 1 label/ total label:')
display(fully_labeled[LABELS].sum(axis=1).value_counts().sort_index().rename('studies').to_frame())

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
label_summary[['negative','positive']].plot.barh(stacked=True, ax=ax, color=['#a9c4d4','#147d92'])
ax.set(xlabel='Labeled studies only', title=f'Labels among {len(fully_labeled)} fully labeled studies')
plt.tight_layout(); plt.show()

In [ ]:
metadata = ['Anatomical_Plane','Fluid_Sensitive','Fat_Suppression']
for n,ss in [('train',series),('test',test_series)]:
    print(n, 'metadata missing:', ss[metadata].isna().sum().to_dict())
    for c in metadata:
        print(c, ss[c].value_counts(dropna=False).to_dict())
    print('Equal value:', bool(ss.Fluid_Sensitive.eq(ss.Fat_Suppression).all()))

series_counts = series.groupby(ID)[SID].nunique().reindex(train[ID], fill_value=0)
display(series_counts.describe().to_frame('series_per_study'))
combinations = series.groupby(metadata, dropna=False).agg(series_count=(SID,'size'), study_count=(ID,'nunique'))
display(combinations)
planes = pd.crosstab(series[ID],series.Anatomical_Plane).reindex(train[ID],fill_value=0)
plane_summary = pd.DataFrame({'series_count': series.Anatomical_Plane.value_counts(),
                              'study_count': planes.gt(0).sum()})
display(plane_summary)

In [ ]:
print('Study with all Axial, Coronal, Sagittal:', int(planes.reindex(columns=['Axial','Coronal','Sagittal'],fill_value=0).gt(0).all(axis=1).sum()))
fig, ax = plt.subplots(figsize=(8,4))
series_counts.value_counts().sort_index().plot.bar(ax=ax, color='#147d92')
ax.set(xlabel='Series per study', ylabel='Studies', title='Train: series per study')
plt.tight_layout(); plt.show()

In [ ]:
study_eda = train[[ID]].copy()
study_eda['label_status'] = status
study_eda['n_series'] = train[ID].map(series_counts)
reports = train.Report.fillna('').astype(str)
study_eda['report_chars'] = reports.str.len()
for plane in ['Axial','Coronal','Sagittal']:
    study_eda['n_'+plane] = train[ID].map(planes[plane])
display(study_eda.groupby('label_status').agg(studies=(ID,'size'),
    series_mean=('n_series','mean'), series_min=('n_series','min'), series_max=('n_series','max'),
    report_chars_median=('report_chars','median')))
for plane in ['Axial','Coronal','Sagittal']:
    display(study_eda.groupby('label_status')['n_'+plane].describe())
print('Labeled study IDs and metadata:')
display(study_eda.loc[study_eda.label_status.eq('fully_labeled')].head())

In [ ]:
normalized = reports.str.replace(r'\s+', ' ', regex=True).str.strip()
print('Empty reports after stripping whitespace:', int(normalized.eq('').sum()))
print('Number of unique verbatim reports:', reports.nunique())
print('Duplicate report rows excluding the first occurrence:', int(reports.duplicated().sum()))
print('Total studies in groups of identical verbatim reports:', int(reports.duplicated(keep=False).sum()))
print('Duplicate report rows after whitespace normalization:', int(normalized.duplicated().sum()))
print('Unique reports shared between labeled and unlabeled groups:',
      len(set(normalized[status.eq('fully_labeled')]) & set(normalized[status.eq('unlabeled')])))
display(reports.str.len().describe().to_frame('report_characters'))
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(reports.str.len(), bins=35, color='#147d92')
ax.set(xlabel='Characters', ylabel='Studies', title='Report length — all train studies')
plt.tight_layout()
plt.show()

In [ ]:
binary = fully_labeled[LABELS].astype(int)
cooccurrence = binary.T.dot(binary)
display(cooccurrence)
fig,ax = plt.subplots(figsize=(9,8))
im=ax.imshow(cooccurrence.to_numpy(),cmap='Blues')
ax.set_xticks(range(len(LABELS)),LABELS,rotation=90)
ax.set_yticks(range(len(LABELS)),LABELS)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        v=int(cooccurrence.iloc[i,j]); ax.text(j,i,str(v),ha='center',va='center',fontsize=8,color='white' if v>22 else '#142b3c')
fig.colorbar(im,ax=ax,label='Studies positive for both labels')
ax.set_title(f'Co-occurrence among {len(binary)} labeled studies')
plt.tight_layout(); plt.show()